In [2]:
!python -V

Python 3.10.6


In [3]:
import pandas as pd

In [4]:
import pickle

In [5]:
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import root_mean_squared_error

In [7]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1717092736403, experiment_id='1', last_update_time=1717092736403, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>

In [8]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [9]:
df_train = read_dataframe('data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('data/green_tripdata_2021-02.parquet')

In [10]:
df_train.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge',
       'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge',
       'duration'],
      dtype='object')

In [11]:
len(df_train), len(df_val)

(73908, 61921)

In [12]:
df_train['PU_DO'] = df_train['PULocationID'].astype(str) + '_' + df_train['DOLocationID'].astype(str)

df_train['PU_DO'] = df_train['PULocationID'].astype(str) + '_' + df_train['DOLocationID'].astype(str)
df_val['PU_DO'] = df_val['PULocationID'].astype(str) + '_' + df_val['DOLocationID'].astype(str)

In [13]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [14]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [15]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

7.758715208946364

In [16]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [17]:
with mlflow.start_run():

    mlflow.set_tag("developer", "rui pinto")

    mlflow.log_param("train-data-path", "data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "data/green_tripdata_2021-02.parquet")

    alpha = 0.01
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [18]:
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

# objective function
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=10
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

# search space
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

# optimization
best_result = fmin(
    fn=objective, # objective function
    space=search_space, # search space
    algo=tpe.suggest, # surrogate algorithm
    max_evals=2, # number of iterations
    trials=Trials() # trials object that keeps track of the sample results
)

  0%|          | 0/2 [00:00<?, ?trial/s, best loss=?]

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [13:17:57] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:10.95160                         
[1]	validation-rmse:9.95588                          
[2]	validation-rmse:9.17806                          
[3]	validation-rmse:8.57571                          
[4]	validation-rmse:8.11193                          
[5]	validation-rmse:7.75995                          
[6]	validation-rmse:7.49515                          
[7]	validation-rmse:7.29220                          
[8]	validation-rmse:7.13825                          
[9]	validation-rmse:7.02430                          
[10]	validation-rmse:6.93660                         
[11]	validation-rmse:6.86896                         
[12]	validation-rmse:6.81396                         
[13]	validation-rmse:6.77164                         
[14]	validation-rmse:6.73770                         
[15]	validation-rmse:6.71387                         
[16]	validation-rmse:6.69361                         
[17]	validation-rmse:6.67824                         
[18]	validation-rmse:6.66572

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [13:18:23] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)



[0]	validation-rmse:7.81028                                                   
[1]	validation-rmse:6.84802                                                   
[2]	validation-rmse:6.64697                                                   
[3]	validation-rmse:6.58394                                                   
[4]	validation-rmse:6.55852                                                   
[5]	validation-rmse:6.54405                                                   
[6]	validation-rmse:6.53796                                                   
[7]	validation-rmse:6.52828                                                   
[8]	validation-rmse:6.52239                                                   
[9]	validation-rmse:6.51487                                                   
[10]	validation-rmse:6.51092                                                  
[11]	validation-rmse:6.50698                                                  
[12]	validation-rmse:6.50218                        

In [19]:
best_result

{'learning_rate': 0.6312126353250297,
 'max_depth': 34.0,
 'min_child_weight': 4.139100439942653,
 'reg_alpha': 0.009393984111075832,
 'reg_lambda': 0.00706854328856213}

In [20]:
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)


with mlflow.start_run():

    best_params = {
        'learning_rate': 0.06549179851966823,
        'max_depth': 65,
        'min_child_weight': 4.730482762129463,
        'objective': 'reg:linear',
        'reg_alpha': 0.009451536770262137,
        'reg_lambda': 0.14515981980661224,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=10
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open('models/preprocessor.b', 'wb') as f_out:
        pickle.dump(dv, f_out)

    mlflow.log_artifact(local_path="models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [13:18:29] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


[0]	validation-rmse:11.68345
[1]	validation-rmse:11.19827
[2]	validation-rmse:10.75456
[3]	validation-rmse:10.34939
[4]	validation-rmse:9.98008
[5]	validation-rmse:9.64399
[6]	validation-rmse:9.33860
[7]	validation-rmse:9.06146
[8]	validation-rmse:8.81024
[9]	validation-rmse:8.58308
[10]	validation-rmse:8.37812
[11]	validation-rmse:8.19259
[12]	validation-rmse:8.02590
[13]	validation-rmse:7.87547
[14]	validation-rmse:7.74046
[15]	validation-rmse:7.61856
[16]	validation-rmse:7.50947
[17]	validation-rmse:7.41096
[18]	validation-rmse:7.32291
[19]	validation-rmse:7.24386
[20]	validation-rmse:7.17312
[21]	validation-rmse:7.10941
[22]	validation-rmse:7.05222
[23]	validation-rmse:7.00106
[24]	validation-rmse:6.95501
[25]	validation-rmse:6.91375
[26]	validation-rmse:6.87678
[27]	validation-rmse:6.84261
[28]	validation-rmse:6.81242
[29]	validation-rmse:6.78495
[30]	validation-rmse:6.76033
[31]	validation-rmse:6.73791
[32]	validation-rmse:6.71790
[33]	validation-rmse:6.69978
[34]	validation-rmse

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [13:19:34] WARNING: /Users/runner/work/xgboost/xgboost/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
  warnings.warn(smsg, UserWarning)


In [21]:
logged_model = 'runs:/0541a276f8fd4f29b599aa669c12447f/models_mlflow'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)

/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/xgboost/core.py:160: UserWarning: [13:19:36] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:209: reg:linear is now deprecated in favor of reg:squarederror.
  warnings.warn(smsg, UserWarning)


In [22]:
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: models_mlflow
  flavor: mlflow.xgboost
  run_id: 0541a276f8fd4f29b599aa669c12447f

In [23]:
xgboost_model = mlflow.xgboost.load_model(logged_model)

In [24]:
xgboost_model

In [25]:
y_pred = xgboost_model.predict(valid)

In [26]:
y_pred[:10]

array([13.868566,  6.923356, 13.369489, 24.996273,  9.37541 , 17.25084 ,
       10.490083,  8.139304,  9.249404, 16.307281], dtype=float32)

In [27]:
# from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
# from sklearn.svm import LinearSVR

# for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

#     with mlflow.start_run():

#         mlflow.log_param("train-data-path", "data/green_tripdata_2021-01.parquet")
#         mlflow.log_param("valid-data-path", "data/green_tripdata_2021-02.parquet")
#         mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

#         mlmodel = model_class()
#         mlmodel.fit(X_train, y_train)

#         y_pred = mlmodel.predict(X_val)
#         rmse = root_mean_squared_error(y_val, y_pred)
#         mlflow.log_metric("rmse", rmse)


In [28]:
from mlflow.tracking import MlflowClient

TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=TRACKING_URI)

In [30]:
client.create_experiment(name="my-cool-experiment-2")

'3'

In [31]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids="1",
    filter_string="metrics.rmse < 13",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

runs

[<Run: data=<RunData: metrics={'rmse': 6.35385320032779}, params={'learning_rate': '0.3886379233814974',
  'max_depth': '23',
  'min_child_weight': '1.0017358151177247',
  'objective': 'reg:linear',
  'reg_alpha': '0.2323207188900079',
  'reg_lambda': '0.05832051700732755',
  'seed': '42'}, tags={'mlflow.runName': 'kindly-steed-72',
  'mlflow.source.name': '/Users/ruifspinto/.pyenv/versions/3.10.6/envs/homework2/lib/python3.10/site-packages/ipykernel_launcher.py',
  'mlflow.source.type': 'LOCAL',
  'mlflow.user': 'ruifspinto',
  'model': 'xgboost'}>, info=<RunInfo: artifact_uri='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1/3fa4ba412d41431c910f495fc7021558/artifacts', end_time=1717427236978, experiment_id='1', lifecycle_stage='active', run_id='3fa4ba412d41431c910f495fc7021558', run_name='kindly-steed-72', run_uuid='3fa4ba412d41431c910f495fc7021558', start_time=1717427228451, status='FINISHED', user_id='ruifspinto'>, inputs=<RunInputs: dataset_inputs=[]>>,
 <R

In [32]:
for run in runs:
    print(run.info.run_id, run.data.metrics['rmse'])

3fa4ba412d41431c910f495fc7021558 6.35385320032779
71e0c95e577f4c35829174608771dfdb 6.371888152078425
cf7964df4c224a348c153f846138176e 6.3863321117583505
0541a276f8fd4f29b599aa669c12447f 6.3863321117583505
fb0db6f9e9c343589b715eec88eeed63 6.3863321117583505


In [33]:
MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [34]:
run_id = "3fa4ba412d41431c910f495fc7021558"

model_uri = f"runs:/{run_id}/models_mlflow"
mlflow.register_model(model_uri, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
Created version '3' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1717503599758, current_stage='None', description=None, last_updated_timestamp=1717503599758, name='nyc-taxi-regressor', run_id='3fa4ba412d41431c910f495fc7021558', run_link=None, source='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1/3fa4ba412d41431c910f495fc7021558/artifacts/models_mlflow', status='READY', status_message=None, tags={}, user_id=None, version=3>

In [35]:
model_uri

'runs:/3fa4ba412d41431c910f495fc7021558/models_mlflow'

In [36]:
run_id_2 = "3fa4ba412d41431c910f495fc7021558"

model_uri_2 = f"runs:/{run_id_2}/models_mlflow"
mlflow.register_model(model_uri_2, name="nyc-taxi-regressor")

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
Created version '4' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1717503603934, current_stage='None', description=None, last_updated_timestamp=1717503603934, name='nyc-taxi-regressor', run_id='3fa4ba412d41431c910f495fc7021558', run_link=None, source='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1/3fa4ba412d41431c910f495fc7021558/artifacts/models_mlflow', status='READY', status_message=None, tags={}, user_id=None, version=4>

In [37]:
from pprint import pprint

for rm in client.search_registered_models():
    pprint(dict(rm), indent=4)

{   'aliases': {},
    'creation_timestamp': 1717496891492,
    'description': None,
    'last_updated_timestamp': 1717503603934,
    'latest_versions': [   <ModelVersion: aliases=[], creation_timestamp=1717503603934, current_stage='None', description=None, last_updated_timestamp=1717503603934, name='nyc-taxi-regressor', run_id='3fa4ba412d41431c910f495fc7021558', run_link=None, source='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1/3fa4ba412d41431c910f495fc7021558/artifacts/models_mlflow', status='READY', status_message=None, tags={}, user_id=None, version=4>,
                           <ModelVersion: aliases=[], creation_timestamp=1717497008593, current_stage='Staging', description=None, last_updated_timestamp=1717497939081, name='nyc-taxi-regressor', run_id='3fa4ba412d41431c910f495fc7021558', run_link=None, source='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1/3fa4ba412d41431c910f495fc7021558/artifacts/models_mlflow', status='READY

In [38]:
model_name = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)


for version in latest_versions:
    print(version.version, version.current_stage)

4 None
2 Staging


/var/folders/hc/lcgvr8h11sq4z32r5nwftth80000gp/T/ipykernel_51652/3062051028.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.13.1/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [39]:
client.transition_model_version_stage(
    name=model_name,
    version=2,
    stage="Staging",
    archive_existing_versions=False
)

/var/folders/hc/lcgvr8h11sq4z32r5nwftth80000gp/T/ipykernel_51652/1872662155.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.13.1/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1717497008593, current_stage='Staging', description=None, last_updated_timestamp=1717503607927, name='nyc-taxi-regressor', run_id='3fa4ba412d41431c910f495fc7021558', run_link=None, source='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1/3fa4ba412d41431c910f495fc7021558/artifacts/models_mlflow', status='READY', status_message=None, tags={}, user_id=None, version=2>

In [40]:
from datetime import datetime
stage="Staging"

date = datetime.today().date()

client.update_model_version(
    name=model_name,
    version=2,
    description=f"The model version {version} was transitioned to {stage} on {date}",
)

<ModelVersion: aliases=[], creation_timestamp=1717497008593, current_stage='Staging', description=('The model version <ModelVersion: aliases=[], '
 "creation_timestamp=1717497008593, current_stage='Staging', description=None, "
 "last_updated_timestamp=1717497939081, name='nyc-taxi-regressor', "
 "run_id='3fa4ba412d41431c910f495fc7021558', run_link=None, "
 "source='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1/3fa4ba412d41431c910f495fc7021558/artifacts/models_mlflow', "
 "status='READY', status_message=None, tags={}, user_id=None, version=2> was "
 'transitioned to Staging on 2024-06-04'), last_updated_timestamp=1717503611536, name='nyc-taxi-regressor', run_id='3fa4ba412d41431c910f495fc7021558', run_link=None, source='/Users/ruifspinto/Desktop/mlops-zoomcamp/02-experiment-tracking/mlruns/1/3fa4ba412d41431c910f495fc7021558/artifacts/models_mlflow', status='READY', status_message=None, tags={}, user_id=None, version=2>